# 미니 장미 생장 단계 판단 — 혼합형 (Colab)

2단계(등급 판단)를 **yolov8m_best 봉오리 위치 → FastSAM 봉오리 마스크 → yolo26m-cls_best 등급 → Claude 검수·누락 추가** 로 나눈 버전입니다. 두 모델은 **이미 학습된 가중치**(`weights/yolov8m_best.pt`, `weights/yolo26m-cls_best.pt`)를 그대로 씁니다(노트북에서 학습하지 않음). 자세한 설명: `HYBRID.md`

**처음 한 번만**
1. `rose_hybrid` 폴더를 구글 드라이브 **내 드라이브** 바로 아래에 통째로 업로드합니다.
2. 드라이브에서 `rose_hybrid.ipynb` 를 더블클릭 → **Google Colaboratory** 로 엽니다.
3. 프로젝트 폴더에 `ANTHROPIC_API_KEY.txt`(Claude API 키 한 줄)를 넣어 둡니다. GPU 는 없어도 됩니다 (있으면 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 로 더 빠름).

그다음 위에서부터 셀을 차례로 실행합니다 (셀 왼쪽 ▶ 또는 `Shift+Enter`).

| 단계 | 내용 | 필요할 때 |
|---|---|---|
| 0 | 준비 (드라이브 연결) | 매번 |
| 1 | 입력 사진 확인·교체 | 매번 |
| 2 | 구멍 위치 보정 | 처음 1회, 카메라를 옮겼을 때 |
| 3 | 구멍 위치·빈 구멍 점수 확인 | 보정 후 |
| 4 | 1단계만 실행 (검출 모델 없이) | 확인용 |
| 5 | 전체 실행 (학습 없이 모델 바로 적용) | 매번 |
| 6 | 결과 내려받기 | 필요할 때 |

## 0. 준비

In [ ]:
# 이 폴더를 업로드한 위치 (다른 곳에 올렸다면 수정)
PROJECT_DIR = '/content/drive/MyDrive/rose_hybrid'

import os, sys, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:          # 로컬 Jupyter 에서는 노트북이 있는 폴더를 사용
    IN_COLAB = False
    PROJECT_DIR = os.path.abspath('.')

os.chdir(PROJECT_DIR)
for p in (PROJECT_DIR, os.path.join(PROJECT_DIR, 'tools')):
    if p not in sys.path:
        sys.path.insert(0, p)

from IPython.display import Image, display

def latest_run():
    runs = sorted(glob.glob('output/2*'))
    return runs[-1] if runs else None

def show(paths, width=900):
    for p in paths:
        print(p)
        display(Image(p, width=width))

print('작업 폴더:', os.getcwd())
print(sorted(os.listdir()))

In [ ]:
# 봉오리 검출(yolov8m_best)·FastSAM·분류(yolo26m-cls_best)용 라이브러리 설치 — 런타임을 새로 시작할 때마다 필요
# YOLO26 은 ultralytics 8.4.0 이상 필요
# anthropic: Claude 검수·누락 추가 (그리고 detector.type: vlm_plant) 에 사용
%pip install -q -U "ultralytics>=8.4.0" anthropic

# 바로 쓰는 학습된 모델: weights/yolov8m_best.pt (봉오리 위치), weights/yolo26m-cls_best.pt (등급)
for w in ('weights/yolov8m_best.pt', 'weights/yolo26m-cls_best.pt'):
    print(w, ':', '있음' if os.path.exists(w) else '없음 — rose_hybrid 폴더에 함께 업로드하세요')

# Claude API 키: 프로젝트 폴더(rose_hybrid)의 ANTHROPIC_API_KEY.txt 에서 읽습니다 (키 sk-ant-... 한 줄).
# 드라이브의 rose_hybrid 폴더에 이 파일을 올려 두세요. 다른 사람과 공유하지 마세요. VLM 을 쓰지 않으면 없어도 됩니다.
from rose_pipeline.api_key import KEY_FILE, read_api_key
try:
    read_api_key()
    print('Claude API 키:', KEY_FILE, '에서 읽음')
except Exception as e:
    print('Claude API 키: 없음 (VLM 사용 안 함) -', e)

In [ ]:
# (선택) 동작 확인: 합성 이미지로 전체 흐름 테스트 — '통과' 가 나오면 정상
!python tests/test_pipeline.py
!python tests/test_hybrid.py

## 1. 입력 사진

`input/CL.jpg`, `input/CR.jpg` 를 사용합니다. 새 사진으로 바꾸려면 `UPLOAD = True` 로 바꾸고 실행한 뒤
**CL.jpg, CR.jpg** 이름의 파일을 선택하세요.

In [ ]:
UPLOAD = False

if UPLOAD and IN_COLAB:
    from google.colab import files
    for name, data in files.upload().items():
        with open(os.path.join('input', name), 'wb') as f:
            f.write(data)
        print('저장:', name)

show(['input/CL.jpg', 'input/CR.jpg'], width=600)

## 2. 구멍 위치 보정 (처음 1회)

사진 위에서 **R1C1 → R1C2 → … → R3C4** 순서로 구멍 중심을 클릭하고 **[저장]** 을 누릅니다.
- 빈 구멍도 보이면 찍습니다 (식물 유무를 자동 판단할 위치).
- 잎에 가려진 구멍은 화분 위치를 짐작해 찍습니다.
- 사진 가장자리 밖 구멍은 **회색 여백**에 짐작해 찍습니다 (그 식물 잎이 이웃 식물로 섞이는 것을 막음).
- 아예 알 수 없는 구멍은 **[건너뛰기]**.
- 이미 저장된 점이 있으면 처음부터 표시됩니다. 점을 드래그해서 고칠 수 있습니다.

In [ ]:
from calibrate_colab import calibrate
calibrate('CL')

In [ ]:
calibrate('CR')

## 3. 구멍 위치·빈 구멍 점수 확인

빨간 원 = 식물, 파란 원 = 빈 구멍. 점수가 `presence.empty_ratio`(기본 0.38) 이상이면 빈 구멍입니다.
판단이 틀리면 `configs/pipeline.yaml` 의 `presence` 값을 조정하세요.

In [ ]:
!python tools/preview_holes.py
show(['output/holes_preview_CL.jpg', 'output/holes_preview_CR.jpg'])

## 4. 1단계만 실행 (검출 모델 없이)

식물 유무와 식물별 분할만 확인합니다. 모든 식물이 `grade_zero` 로 나옵니다.

In [ ]:
!python run_pipeline.py --no-detector
show(sorted(glob.glob(f'{latest_run()}/vis_*.jpg')))

## 5. 전체 실행 (학습된 모델 바로 적용)

결과는 `output/<시각>/` 에 저장됩니다: `result_buds.jpg`, `result.json`, `table.txt`, `vis_CL.jpg`, `vis_CR.jpg`, `masks/`.
표에서 식물 = 등급, 빈 구멍 = `-`, 판단 불가 = `?`.

봉오리 표시: 박스 = 위치, **흰 윤곽선 = FastSAM 봉오리 마스크**, `G2 0.83 -> R1C1` = 분류 등급 · 분류 확신 · 배정된 칸.
`result.json` 의 봉오리마다 `det_conf`(① 위치 신뢰도)와 `mask_px`(마스크 픽셀 수)가 함께 저장됩니다.

**`result_buds.jpg` = 최종 결과판**: 칸마다 최종 등급과 그 칸에 배정된 봉오리 사진을 높은 등급부터 보여 줍니다. `table.txt` 에는 카메라별 봉오리 개수가 함께 저장됩니다.

**2단계 판단 방식** = `configs/pipeline.yaml` 의 `grading.detector.type`
- `hybrid` (현재 기본): `weights/yolov8m_best.pt` 가 봉오리 위치 → FastSAM → `weights/yolo26m-cls_best.pt` 가 등급 → **Claude 가 검수**해 최종 등급을 정하고, 식물마다 **놓친 봉오리를 추가**합니다(`vis` 에 `+VLM`). `ANTHROPIC_API_KEY.txt` 가 필요합니다(요금, 한 번 실행에 약 20번 요청). `vis_*.jpg` 에서 검수로 바뀐 등급은 `(cls G4)` 처럼 분류 모델 등급이 괄호로 붙습니다. Claude 없이 모델만 쓰려면 설정에서 `review: false`, `add_missed: false`.
- `vlm_plant`: 식물(화분)마다 그 식물 사진을 Claude 에게 보내 봉오리·꽃 위치와 등급을 직접 받습니다. 학습이 필요 없지만 **Claude API 키 파일 `ANTHROPIC_API_KEY.txt` 가 필요**하고(0번 설치 셀) 실행할 때마다 요금이 듭니다(식물 수 × 카메라 수 = 약 12번 요청). 끝나면 `VLM 판단` 표(칸·카메라별 요약)와 `VLM 사용량` 이 출력되고, `result_buds.jpg` 칸 제목 아래에 VLM 요약, `vlm/` 폴더에 VLM 에 보낸 식물 사진과 답이 저장됩니다.
- 방식 바꾸기: `configs/pipeline.yaml` 의 `grading.detector.type`·`cls_backend`·`review`·`add_missed` 를 고칩니다.

어느 방식이든 그 뒤(식물 등급 = 봉오리 중 최고 등급 → 카메라 합치기 → 표·결과판)와 결과 파일 양식은 같습니다. `vlm_plant` 는 FastSAM 마스크가 없어 흰 윤곽선이 그려지지 않고, `result.json` 에 `det_conf`·`mask_px` 가 없습니다.

In [ ]:
!python run_pipeline.py
show([f'{latest_run()}/result_buds.jpg'], width=1100)
show(sorted(glob.glob(f'{latest_run()}/vis_*.jpg')))

# Claude 누락 추가(또는 vlm_plant): Claude 에 보낸 식물 사진 + 답한 박스와 요약 (칸·카메라마다 1장)
show(sorted(glob.glob(f'{latest_run()}/vlm/*.jpg')), width=450)

## 6. 결과 내려받기

In [ ]:
import shutil

run = latest_run()
if run:
    base = '/content' if IN_COLAB else 'output'
    archive = shutil.make_archive(os.path.join(base, os.path.basename(run)), 'zip', run)
    print('압축:', archive)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)